# Estimasi Usia dari Citra Wajah
## UTKFace dan MobileNetV3 Small

Notebook proyek visi komputer: audit data, split tetap, baseline, pelatihan,
evaluasi, visualisasi, dan inferensi. Implementasi menggunakan modul `faceage/`
di repo ini agar hasil notebook dan aplikasi konsisten.

**Status:** belum ada hasil eksperimen UTKFace. Angka penelitian hanya muncul
setelah menjalankan pelatihan dengan data nyata. Logbook di `docs/logbook_kosong.csv`
tetap kosong dan diisi sendiri.

### Cara membuka
1. Buka notebook dari folder repo menggunakan VS Code dengan dukungan Jupyter, atau Jupyter.
2. Buat `.venv` sesuai README bila belum ada, lalu pasang dependensi:
   `.\.venv\Scripts\python.exe -m pip install -r requirements-notebook.txt`.
3. Pilih kernel Python `.venv` repo ini. Jalankan sel dari atas ke bawah.
4. Unduh [UTKFace Aligned & Cropped dari Kaggle](https://www.kaggle.com/datasets/moritzm00/utkface-cropped),
   salinan pihak ketiga; login jika diminta, lalu pilih Download dan ekstrak citra
   ke `data/raw/UTKFace/`. Pertahankan nama file asli.
   Rujukan pengembang: [situs resmi](https://susanqq.github.io/UTKFace/)
   dan [repo AICIP](https://github.com/aicip/UTKFace).
   Tautan Google Drive resmi cropped dan repo GitHub lama `susanqq/UTKFace`
   mengembalikan 404 saat diperiksa pada 6 Oktober 2026. Halaman Kaggle dapat
   dibuka; arsipnya belum diunduh atau diverifikasi di repo ini.
   Catat URL dan versi arsip yang benar-benar dipakai untuk eksperimen.
5. Aktifkan flag tiap tahap yang hendak dijalankan pada sel konfigurasi.

Notebook belum berisi output tersimpan. Menjalankan semua sel dengan flag default
hanya memeriksa lingkungan dan menampilkan petunjuk; tidak memulai pelatihan panjang.

## 1. Lingkungan dan modul proyek
Gunakan folder repo sebagai working directory. Untuk lokasi lain, sesuaikan `ROOT`.
Bobot ImageNet diunduh ke cache lokal repo saat pertama diperlukan.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'faceage').is_dir():
    raise RuntimeError('Buka notebook dari folder ComVis atau sesuaikan ROOT ke folder repo.')
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
os.environ.setdefault('TORCH_HOME', str(ROOT / '.cache' / 'torch'))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache' / 'matplotlib'))

import torch
from IPython.display import display, Image as DisplayImage
from faceage.data import prepare_data, load_manifest, manifest_fingerprint
from faceage.io import read_json, read_csv
from faceage.training import train, validate_config
from faceage.evaluation import compare_runs
from faceage.inference import Predictor, decode_image

print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
print('Repo:', ROOT)

## 2. Konfigurasi eksperimen
Split train/validation/test menargetkan 70/15/15 dengan seed 42; duplikat berada
dalam kelompok yang sama. Rasio aktual bergantung pada kelompok data.

Aktifkan `RUN_PREPARE` untuk audit awal, lalu `RUN_TRAINING` untuk menjalankan
sel B0, B1, M1, dan B1 control. Gunakan `RUN_PREFIX` baru untuk eksperimen baru;
folder hasil lama tidak ditimpa. `RUN_COMPARE` menghitung ulang evaluasi validasi.
`RUN_FINAL_TEST` hanya diaktifkan setelah keputusan model dan konfigurasi dibekukan.

Parameter default: RGB 224×224, AdamW, Smooth L1, maksimum 30 epoch per tahap,
batch 32, patience 5. Learning rate head 1e-4 dan backbone 1e-5.

In [ ]:
DATA = ROOT / 'data/raw/UTKFace'
PROCESSED = ROOT / 'data/processed'
MANIFEST = PROCESSED / 'manifest.csv'
RUN_PREFIX = 'notebook01'
DEVICE = 'auto'  # auto / cpu / cuda
CONFIG = validate_config(read_json(ROOT / 'configs/default.json'))
RUNS = {method: ROOT / 'runs' / f'{RUN_PREFIX}_{method}'
        for method in ('b0', 'b1', 'm1', 'b1_control')}
COMPARISON = ROOT / 'runs' / f'{RUN_PREFIX}_comparison'

RUN_PREPARE = False
RUN_TRAINING = False
RUN_COMPARE = False
RUN_FINAL_TEST = False
RUN_SMOKE = False

print(json.dumps(CONFIG, indent=2))

## 3. Audit data dan manifest
Audit memeriksa nama file/usia, gambar rusak, dimensi, duplikat piksel, dan kemiripan
dHash. Tinjau `invalid_files.csv` dan `near_duplicates.csv`. Kemiripan dHash tidak
menjamin identitas orang terpisah antar-split. Manifest tidak ditimpa otomatis.

In [ ]:
if RUN_PREPARE:
    audit = prepare_data(DATA, PROCESSED, seed=CONFIG['seed'], near_threshold=4)
    display(audit)
elif (PROCESSED / 'audit.json').is_file():
    display(read_json(PROCESSED / 'audit.json'))
else:
    print('Belum ada audit. Unduh dataset, lalu aktifkan RUN_PREPARE.')

rows = load_manifest(MANIFEST) if MANIFEST.is_file() else []
if rows:
    print('Jumlah citra:', len(rows))
    print('Fingerprint:', manifest_fingerprint(rows))
    print('Split:', {name: sum(row['split'] == name for row in rows)
                     for name in ('train', 'validation', 'test')})

## 4. Distribusi usia data
Periksa kelompok usia yang sedikit sebelum memilih eksperimen. Sel ini membaca
manifest tanpa menampilkan atau menyimpan ulang wajah dataset.

In [ ]:
if rows:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    for split in ('train', 'validation', 'test'):
        ages = [row['age'] for row in rows if row['split'] == split]
        ax.hist(ages, bins=list(range(0, 121, 5)), alpha=.45, label=split)
    ax.set(xlabel='Label usia (tahun)', ylabel='Jumlah citra', title='Distribusi label UTKFace')
    ax.legend()
    display(fig)
    plt.close(fig)
else:
    print('Grafik tersedia setelah manifest dibuat.')

## 5. Baseline B0 — median usia data latih
B0 tidak menggunakan citra. Median dihitung hanya dari data latih, dan dipakai
untuk mengetahui apakah model citra lebih baik daripada prediksi konstan.

In [ ]:
if RUN_TRAINING:
    display(train('b0', DATA, MANIFEST, RUNS['b0'], CONFIG, DEVICE))
else:
    print('B0 belum dijalankan pada sesi ini; aktifkan RUN_TRAINING.')

## 6. Baseline B1 — fitur pralatih dengan backbone beku
MobileNetV3 Small menggunakan ImageNet. Backbone beserta statistik BatchNorm
dibekukan; classifier/head regresi dilatih. Checkpoint terbaik dipilih berdasarkan
MAE validasi, bukan hasil test set.

In [ ]:
if RUN_TRAINING:
    display(train('b1', DATA, MANIFEST, RUNS['b1'], CONFIG, DEVICE))
else:
    print('B1 belum dijalankan pada sesi ini.')

## 7. Metode M1 — fine tuning
M1 dimulai dari checkpoint B1 pada split dan konfigurasi yang sama. Tiga modul
terakhir backbone serta head dibuka. Sel ini membutuhkan B1 selesai terlebih dahulu.

In [ ]:
if RUN_TRAINING:
    display(train('m1', DATA, MANIFEST, RUNS['m1'], CONFIG, DEVICE,
                  init_checkpoint=RUNS['b1'] / 'best.pt'))
else:
    print('M1 belum dijalankan pada sesi ini.')

## 8. Kontrol tambahan — B1 control
B1 control dimulai dari checkpoint B1 yang sama dengan M1, lalu meneruskan
pelatihan head dengan backbone tetap beku. Perbandingan ini membantu menilai
perubahan fine tuning setelah kedua model memperoleh tahap pelatihan tambahan.
Batas epoch sama; early stopping dapat membuat jumlah epoch aktual berbeda.

In [ ]:
if RUN_TRAINING:
    display(train('b1_control', DATA, MANIFEST, RUNS['b1_control'], CONFIG, DEVICE,
                  init_checkpoint=RUNS['b1'] / 'best.pt'))
else:
    print('B1 control belum dijalankan pada sesi ini.')

## 9. Evaluasi dan perbandingan validasi
Laporkan MAE, RMSE, median absolute error, MAE per kelompok usia, serta jumlah
data tiap kelompok. Target penurunan MAE 20% terhadap B0 adalah sasaran proposal,
bukan hasil yang diasumsikan. Semua model menggunakan manifest yang sama.

In [ ]:
if RUN_COMPARE:
    comparison = compare_runs(list(RUNS.values()), DATA, MANIFEST, COMPARISON,
                              split='validation', device=DEVICE)
    display(comparison)
else:
    print('Aktifkan RUN_COMPARE setelah seluruh model selesai dilatih.')

for method, run in RUNS.items():
    scores = run / 'evaluation_validation' / 'metrics.json'
    if scores.is_file():
        print(method.upper())
        display(read_json(scores))

## 10. Grafik dan analisis kasus
Grafik menggunakan prediksi mentah. Kasus gagal disimpan sebagai path dan galat,
bukan alasan kegagalan yang dikarang. Tinjau citra secara lokal, lalu isi analisis
manusia pada `docs/ERROR_ANALYSIS.md`.

In [ ]:
SELECTED_METHOD = 'm1'
evaluation_dir = RUNS[SELECTED_METHOD] / 'evaluation_validation'
for filename in ('learning_curve.png', 'prediction_vs_label.png',
                 'residual_histogram.png', 'age_group_mae.png'):
    path = evaluation_dir / filename
    if path.is_file():
        print(filename)
        display(DisplayImage(filename=str(path)))
worst = evaluation_dir / 'worst_examples.csv'
if worst.is_file():
    display(read_csv(worst)[:10])
else:
    print('Grafik dan kasus gagal belum tersedia untuk model terpilih.')

## 11. Evaluasi akhir test set
Setelah konfigurasi dipilih menggunakan validasi dan dibekukan, aktifkan
`RUN_FINAL_TEST`. Jangan memilih model atau menyetel hyperparameter berdasarkan
skor test. Evaluasi akhir seluruh metode dilakukan pada split test yang sama.

In [ ]:
if RUN_FINAL_TEST:
    display(compare_runs(list(RUNS.values()), DATA, MANIFEST, COMPARISON,
                         split='test', device=DEVICE, final_test=True))
else:
    print('Evaluasi test belum dijalankan; RUN_FINAL_TEST = False.')

## 12. Inferensi satu wajah baru
Isi `IMAGE_PATH` dengan foto JPG/PNG satu wajah yang sudah dipotong. Tidak ada
deteksi/crop otomatis. Gunakan foto dengan izin. Model penelitian terlatih diperlukan;
checkpoint sintetis ditolak untuk estimasi usia. Pembulatan/pembatasan tampilan
tidak digunakan saat menghitung metrik penelitian.

In [ ]:
IMAGE_PATH = None  # Contoh: ROOT / 'data/local/cropped_face.jpg'
CHECKPOINT = RUNS[SELECTED_METHOD] / 'best.pt'
if IMAGE_PATH is not None:
    image = decode_image(Path(IMAGE_PATH).read_bytes())
    display(image)
    predictor = Predictor(CHECKPOINT, device=DEVICE)
    display(predictor.predict(image))
else:
    print('Isi IMAGE_PATH setelah checkpoint model penelitian tersedia.')

## 13. Uji teknis opsional tanpa dataset wajah
Aktifkan `RUN_SMOKE` untuk menjalankan pipeline pada 30 gambar acak dan empat
metode satu epoch. Ini **bukan hasil estimasi usia wajah** atau bukti akurasi UTKFace.
Hasil disimpan di folder lokal yang diabaikan Git. Setiap pengulangan memakai
folder baru; tidak menimpa bukti lama.

In [ ]:
if RUN_SMOKE:
    import subprocess
    import uuid
    smoke_output = ROOT / 'artifacts/local' / f'notebook_smoke_{uuid.uuid4().hex[:8]}'
    subprocess.run([sys.executable, str(ROOT / 'scripts/smoke_test.py'),
                    '--output', str(smoke_output)], cwd=ROOT, check=True)
    display(read_json(smoke_output / 'smoke_summary.json'))
else:
    print('Smoke sintetis tidak dijalankan pada sesi ini.')

## 14. Demo, dokumentasi, dan logbook
Jalankan demo dari terminal di folder repo:

```powershell
.\.venv\Scripts\python.exe -m streamlit run app.py
```

- [README](README.md): perintah CLI, konfigurasi, dan langkah reproduksi.
- [Dataset](docs/DATASET.md): sumber, label, dan ketentuan penggunaan.
- [Referensi](docs/REFERENCES.md): lima rujukan kredibel, termasuk paper metode.
- [Logbook CSV kosong](docs/logbook_kosong.csv) dan [Markdown kosong](docs/LOGBOOK.md).
- [Pemetaan panduan UTS/UAS](docs/PANDUAN_UTS_UAS.md).

Label UTKFace berasal dari estimasi algoritme serta pemeriksaan manusia. MAE
mengukur kesesuaian terhadap label tersebut; bukan pengukuran usia kronologis
yang terverifikasi. Analisis generalisasi memerlukan data di luar dataset latih.

Notebook dan modul ditulis dengan bantuan OpenAI Codex sesuai proposal pengguna.
Kesimpulan penelitian, kontribusi anggota, dan logbook diisi berdasarkan pekerjaan
serta hasil nyata; notebook tidak mengisinya otomatis.